# Plant Disease Multi-Class Classification — Full Pipeline

**Architecture**: Custom Inception-cell + DenseNet-style SkipConnection hybrid  
**Classes**: 24 (Cotton ×6 · Wheat ×3 · Potato ×3 · Bell Pepper ×2 · Tomato ×10)  
**Input**: 256×256 RGB · **Batch**: 64 · **Optimizer**: Adam + Cosine Annealing LR  

| Step | Description |
|------|-------------|
| 1 | Kaggle API + dataset download |
| 2 | Directory exploration & class mapping |
| 3 | Unified 24-class dataset construction |
| 4 | EDA — class distribution & sample images |
| 5 | Train / Validation / Test split (70 / 15 / 15) |
| 6 | `tf.data` pipeline with augmentation |
| 7 | Custom `Inception_cell` + `SkipConnection` layers |
| 8 | Training — CosineAnnealing + EarlyStopping |
| 9 | Accuracy · Precision · Recall · F1 · Confusion Matrix |

> **GPU recommended.** Open in Colab: Runtime → Change runtime type → T4 GPU


In [ ]:
# Install all required packages
!pip install kaggle tensorflow scikit-learn matplotlib seaborn pandas numpy Pillow tqdm -q
print("All packages installed successfully.")


In [ ]:
import os, math, random, json, warnings, zipfile, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from collections import defaultdict
from matplotlib.patches import Patch
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Flatten, Dense, Dropout, Add, Concatenate, Layer
)
from tensorflow.keras.callbacks import (
    LearningRateScheduler, EarlyStopping, ModelCheckpoint
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs found : {len(gpus)}')
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
    print(f'  {g.name}')
if not gpus:
    print('  No GPU — training will be slow. Use Colab T4 runtime.')


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR    = os.path.join(os.path.expanduser('~'), 'plant_disease_project')
RAW_DIR     = os.path.join(BASE_DIR, 'raw')
UNIFIED_DIR = os.path.join(BASE_DIR, 'unified')
CKPT_DIR    = os.path.join(BASE_DIR, 'checkpoints')
LOG_DIR     = os.path.join(BASE_DIR, 'logs')

for d in [BASE_DIR, RAW_DIR, UNIFIED_DIR, CKPT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Hyperparameters ─────────────────────────────────────────────────────────
IMG_SIZE     = (256, 256)
IMG_SHAPE    = (256, 256, 3)
BATCH_SIZE   = 64
NUM_CLASSES  = 24
INITIAL_LR   = 1e-3
EPOCHS       = 50
DROPOUT_RATE = 0.5
TRAIN_SPLIT  = 0.70
VAL_SPLIT    = 0.15
TEST_SPLIT   = 0.15
SEED         = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── 24 Class Definitions ────────────────────────────────────────────────────
TARGET_CLASSES = [
    'Cotton_Aphids', 'Cotton_Army_worm', 'Cotton_Bacterial_Blight',
    'Cotton_Healthy', 'Cotton_Powdery_mildew', 'Cotton_Target_Spot',
    'Wheat_Brown_rust', 'Wheat_Healthy', 'Wheat_Yellow_rust',
    'Potato_Early_blight', 'Potato_Late_blight', 'Potato_Healthy',
    'Bell_Pepper_Bacterial_spot', 'Bell_Pepper_Healthy',
    'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight',
    'Tomato_Leaf_mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites',
    'Tomato_Target_spot', 'Tomato_Yellowleaf_curl_virus',
    'Tomato_Mosaic_virus', 'Tomato_Healthy',
]
CLASS_TO_IDX = {cls: i for i, cls in enumerate(TARGET_CLASSES)}
IDX_TO_CLASS = {i: cls for i, cls in enumerate(TARGET_CLASSES)}

# Target sample counts (from paper table)
TARGET_COUNTS = {
    'Cotton_Aphids':800,'Cotton_Army_worm':800,'Cotton_Bacterial_Blight':800,
    'Cotton_Healthy':800,'Cotton_Powdery_mildew':800,'Cotton_Target_Spot':788,
    'Wheat_Brown_rust':902,'Wheat_Healthy':1116,'Wheat_Yellow_rust':924,
    'Potato_Early_blight':1000,'Potato_Late_blight':1000,'Potato_Healthy':152,
    'Bell_Pepper_Bacterial_spot':997,'Bell_Pepper_Healthy':1478,
    'Tomato_Bacterial_spot':2127,'Tomato_Early_blight':1000,
    'Tomato_Late_blight':1909,'Tomato_Leaf_mold':952,
    'Tomato_Septoria_leaf_spot':1771,'Tomato_Spider_mites':1676,
    'Tomato_Target_spot':1404,'Tomato_Yellowleaf_curl_virus':3209,
    'Tomato_Mosaic_virus':373,'Tomato_Healthy':1591,
}

VALID_EXTS = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}

def get_images(folder):
    """Return full paths of all image files in a folder."""
    if not os.path.isdir(folder):
        return []
    return [
        os.path.join(folder, f) for f in os.listdir(folder)
        if os.path.splitext(f)[1].lower() in VALID_EXTS
    ]

print('Configuration ready.')
print(f'  Base directory : {BASE_DIR}')
print(f'  Image size     : {IMG_SIZE}')
print(f'  Batch size     : {BATCH_SIZE}')
print(f'  Epochs         : {EPOCHS}')
print(f'  Classes        : {NUM_CLASSES}')


## Step 1 — Kaggle API Setup

1. Go to **https://www.kaggle.com/settings** → **API** → **Create New Token**
2. A `kaggle.json` file will be downloaded to your machine
3. Run the cell below — on **Colab** it prompts a file upload; on **local** it reads from `~/.kaggle/kaggle.json`

**Datasets used:**
- `mohitsingh1804/plantvillage` — Tomato, Potato, Bell Pepper (PlantVillage)
- `dhamur/cotton-plant-disease` — Cotton diseases
- `sinadunk23/behzad-safari-jalal` — Wheat diseases


In [ ]:
def setup_kaggle():
    kaggle_dir  = os.path.expanduser('~/.kaggle')
    kaggle_json = os.path.join(kaggle_dir, 'kaggle.json')
    try:
        from google.colab import files
        print('Upload your kaggle.json file now:')
        uploaded = files.upload()
        os.makedirs(kaggle_dir, exist_ok=True)
        for fname, content in uploaded.items():
            with open(kaggle_json, 'wb') as f:
                f.write(content)
        os.chmod(kaggle_json, 0o600)
        print(f'Saved kaggle.json → {kaggle_json}')
    except ImportError:
        if os.path.exists(kaggle_json):
            print(f'kaggle.json found at {kaggle_json}')
        else:
            raise FileNotFoundError(
                f'Place kaggle.json at {kaggle_json}\n'
                'Get it from: https://www.kaggle.com/settings → API → Create New Token'
            )

setup_kaggle()


In [ ]:
# ── Download all three datasets ──────────────────────────────────────────
KAGGLE_DATASETS = [
    ('mohitsingh1804/plantvillage',    'plantvillage'),
    ('dhamur/cotton-plant-disease',    'cotton-plant-disease'),
    ('sinadunk23/behzad-safari-jalal', 'behzad-safari-jalal'),
]

orig_dir = os.getcwd()
os.chdir(RAW_DIR)

for ds_slug, folder_name in KAGGLE_DATASETS:
    target_folder = os.path.join(RAW_DIR, folder_name)
    zip_file      = os.path.join(RAW_DIR, folder_name + '.zip')

    if os.path.isdir(target_folder) and os.listdir(target_folder):
        print(f'  Already ready : {folder_name}/')
        continue

    if not os.path.isfile(zip_file):
        print(f'Downloading {ds_slug} ...')
        ret = os.system(f'kaggle datasets download -d {ds_slug} --quiet')
        if ret != 0:
            print(f'  ERROR downloading {ds_slug} — check credentials.')
            continue

    print(f'Extracting {folder_name}.zip ...')
    with zipfile.ZipFile(zip_file, 'r') as zf:
        zf.extractall(target_folder)
    print(f'  Done → {target_folder}/')

os.chdir(orig_dir)
print('\nAll datasets downloaded and extracted.')


In [ ]:
# ── Explore directory trees so we know exact folder names ─────────────────
def explore_tree(root, max_depth=5, _depth=0, _prefix=''):
    if _depth > max_depth or not os.path.isdir(root):
        return
    entries = sorted(os.listdir(root))
    for i, entry in enumerate(entries):
        path = os.path.join(root, entry)
        if not os.path.isdir(path):
            continue
        is_last = (i == len(entries) - 1)
        connector = '└── ' if is_last else '├── '
        imgs = [f for f in os.listdir(path)
                if os.path.splitext(f)[1].lower() in VALID_EXTS]
        tag  = f'  [{len(imgs)} imgs]' if imgs else ''
        print(f'{_prefix}{connector}{entry}/{tag}')
        if not imgs:
            ext = '    ' if is_last else '│   '
            explore_tree(path, max_depth, _depth+1, _prefix+ext)

for _, folder in KAGGLE_DATASETS:
    root = os.path.join(RAW_DIR, folder)
    print(f'\n{folder}/')
    if os.path.isdir(root):
        explore_tree(root, max_depth=4)
    else:
        print('  NOT FOUND — check download step above')


In [ ]:
# ── Folder-name → target class  (lowercase keys) ─────────────────────────
#
# If the explore output above shows different folder names,
# add them here following the same pattern.
FOLDER_MAP = {
    # ─── Cotton (from cotton-plant-disease) ───────────────────────────────
    'aphids'                                      : 'Cotton_Aphids',
    'aphid'                                       : 'Cotton_Aphids',
    'army worm'                                   : 'Cotton_Army_worm',
    'army_worm'                                   : 'Cotton_Army_worm',
    'armyworm'                                    : 'Cotton_Army_worm',
    'bacterial blight'                            : 'Cotton_Bacterial_Blight',
    'bacterial_blight'                            : 'Cotton_Bacterial_Blight',
    'powdery mildew'                              : 'Cotton_Powdery_mildew',
    'powdery_mildew'                              : 'Cotton_Powdery_mildew',
    'powderymildew'                               : 'Cotton_Powdery_mildew',
    'target spot'                                 : 'Cotton_Target_Spot',
    'target_spot'                                 : 'Cotton_Target_Spot',
    # ─── Bell Pepper (PlantVillage naming variants) ───────────────────────
    'pepper,_bell___bacterial_spot'               : 'Bell_Pepper_Bacterial_spot',
    'pepper___bacterial_spot'                     : 'Bell_Pepper_Bacterial_spot',
    'bell_pepper___bacterial_spot'                : 'Bell_Pepper_Bacterial_spot',
    'bell pepper___bacterial_spot'                : 'Bell_Pepper_Bacterial_spot',
    'pepper,_bell___healthy'                      : 'Bell_Pepper_Healthy',
    'pepper___healthy'                            : 'Bell_Pepper_Healthy',
    'bell_pepper___healthy'                       : 'Bell_Pepper_Healthy',
    'bell pepper___healthy'                       : 'Bell_Pepper_Healthy',
    # ─── Potato ───────────────────────────────────────────────────────────
    'potato___early_blight'                       : 'Potato_Early_blight',
    'potato___late_blight'                        : 'Potato_Late_blight',
    'potato___healthy'                            : 'Potato_Healthy',
    # ─── Tomato ───────────────────────────────────────────────────────────
    'tomato___bacterial_spot'                     : 'Tomato_Bacterial_spot',
    'tomato___early_blight'                       : 'Tomato_Early_blight',
    'tomato___late_blight'                        : 'Tomato_Late_blight',
    'tomato___leaf_mold'                          : 'Tomato_Leaf_mold',
    'tomato___septoria_leaf_spot'                 : 'Tomato_Septoria_leaf_spot',
    'tomato___spider_mites two-spotted_spider_mite':'Tomato_Spider_mites',
    'tomato___spider_mites_two-spotted_spider_mite':'Tomato_Spider_mites',
    'tomato___spider_mites'                       : 'Tomato_Spider_mites',
    'tomato___target_spot'                        : 'Tomato_Target_spot',
    'tomato___tomato_yellow_leaf_curl_virus'       : 'Tomato_Yellowleaf_curl_virus',
    'tomato___tomato_mosaic_virus'                : 'Tomato_Mosaic_virus',
    'tomato___healthy'                            : 'Tomato_Healthy',
    # ─── Wheat (behzad-safari-jalal) ─────────────────────────────────────
    'brown rust'                                  : 'Wheat_Brown_rust',
    'brown_rust'                                  : 'Wheat_Brown_rust',
    'brownrust'                                   : 'Wheat_Brown_rust',
    'yellow rust'                                 : 'Wheat_Yellow_rust',
    'yellow_rust'                                 : 'Wheat_Yellow_rust',
    'yellowrust'                                  : 'Wheat_Yellow_rust',
}

# 'Healthy' / 'Normal' folders → resolved by parent path context
HEALTHY_CONTEXT = {
    'cotton' : 'Cotton_Healthy',
    'wheat'  : 'Wheat_Healthy',
    'potato' : 'Potato_Healthy',
}

def resolve_class(folder_path):
    name = os.path.basename(folder_path).lower().strip()
    if name in FOLDER_MAP:
        return FOLDER_MAP[name]
    # context-based healthy resolution
    if name in ('healthy', 'normal', 'disease free'):
        full_lower = folder_path.lower()
        for ctx_key, cls in HEALTHY_CONTEXT.items():
            if ctx_key in full_lower:
                return cls
    return None

print('Class mapping defined.')
print(f'  FOLDER_MAP entries : {len(FOLDER_MAP)}')


In [ ]:
# ── Walk all three dataset roots and collect image paths per class ─────────
class_files = defaultdict(list)
unmatched   = []

for _, folder in KAGGLE_DATASETS:
    ds_root = os.path.join(RAW_DIR, folder)
    if not os.path.isdir(ds_root):
        print(f'SKIP (not found): {ds_root}')
        continue
    for dirpath, dirnames, filenames in os.walk(ds_root):
        imgs = [f for f in filenames
                if os.path.splitext(f)[1].lower() in VALID_EXTS]
        if not imgs:
            continue
        cls = resolve_class(dirpath)
        if cls:
            class_files[cls].extend(
                [os.path.join(dirpath, f) for f in imgs]
            )
        else:
            unmatched.append(os.path.basename(dirpath))

print('Scan results:')
all_ok = True
for cls in TARGET_CLASSES:
    n   = len(class_files[cls])
    tgt = TARGET_COUNTS[cls]
    ok  = 'OK ' if n > 0 else 'MISSING'
    print(f'  {ok}  {cls:<40}  found={n:>5}  target={tgt:>5}')
    if n == 0:
        all_ok = False

if not all_ok:
    print('\nWARNING: Some classes have 0 images.')
    print('Check the explore output above and add any missing folder names to FOLDER_MAP.')
    print('Unmatched folders detected:', set(unmatched))
else:
    print('\nAll 24 classes found!')


In [ ]:
# ── Copy images into a clean unified directory (one sub-folder per class) ──
random.seed(SEED)
actual_counts = {}

print('Building unified dataset ...')
for cls in TARGET_CLASSES:
    cls_dir = os.path.join(UNIFIED_DIR, cls)
    os.makedirs(cls_dir, exist_ok=True)

    existing = get_images(cls_dir)
    if existing:
        actual_counts[cls] = len(existing)
        continue

    srcs = class_files[cls][:]
    random.shuffle(srcs)
    srcs = srcs[:TARGET_COUNTS[cls]]

    for i, src in enumerate(srcs):
        ext = os.path.splitext(src)[1].lower() or '.jpg'
        dst = os.path.join(cls_dir, f'{i:05d}{ext}')
        if not os.path.exists(dst):
            shutil.copy2(src, dst)

    actual_counts[cls] = len(srcs)

total = sum(actual_counts.values())
print(f'\nUnified dataset ready!')
print(f'  Total images : {total:,}')
print(f'  Classes      : {len(actual_counts)}')
print(f'  Location     : {UNIFIED_DIR}')
for cls in TARGET_CLASSES:
    print(f'    {cls:<40} {actual_counts[cls]:>5}')


In [ ]:
# ── Class Distribution Chart ───────────────────────────────────────────────
plant_colors = {
    'Cotton':'#E8845A','Wheat':'#F5C842','Potato':'#7EC8A4',
    'Bell_Pepper':'#C084D4','Tomato':'#E86B6B'
}

def plant_of(cls):
    for p in plant_colors:
        if cls.startswith(p):
            return p
    return 'Tomato'

classes = TARGET_CLASSES
counts  = [actual_counts[c] for c in classes]
colors  = [plant_colors[plant_of(c)] for c in classes]
short   = [c.replace('_',' ').replace('Bell Pepper','BP') for c in classes]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].barh(range(len(classes)), counts, color=colors, edgecolor='white', lw=0.5)
axes[0].set_yticks(range(len(classes)))
axes[0].set_yticklabels(short, fontsize=9)
axes[0].set_xlabel('Number of Images', fontsize=11)
axes[0].set_title('Class Distribution', fontweight='bold', fontsize=12)
axes[0].axvline(np.mean(counts), color='#333', ls='--', lw=1, alpha=0.5,
                label=f'Mean: {int(np.mean(counts))}')
axes[0].legend(fontsize=10)
axes[0].grid(True, axis='x', alpha=0.3)

plant_totals = defaultdict(int)
for c, n in actual_counts.items():
    plant_totals[plant_of(c)] += n

axes[1].pie(
    plant_totals.values(),
    labels=[f"{k}\n({v:,})" for k, v in plant_totals.items()],
    colors=[plant_colors[p] for p in plant_totals],
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor':'white','linewidth':1.5}
)
axes[1].set_title('Distribution by Plant Type', fontweight='bold', fontsize=12)

fig.suptitle(f'Plant Disease Dataset  —  {sum(counts):,} images · {len(classes)} classes',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Sample Images Grid (one per class) ────────────────────────────────────
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
axes_flat = axes.flatten()

for i, cls in enumerate(TARGET_CLASSES):
    imgs = get_images(os.path.join(UNIFIED_DIR, cls))
    ax   = axes_flat[i]
    if imgs:
        img = Image.open(random.choice(imgs)).convert('RGB').resize((150, 150))
        ax.imshow(img)
    ax.set_title(cls.replace('_',' '), fontsize=7, pad=2,
                 color=plant_colors[plant_of(cls)])
    ax.axis('off')
    for sp in ax.spines.values():
        sp.set_edgecolor(plant_colors[plant_of(cls)])
        sp.set_linewidth(2)

fig.suptitle('Sample Images — One per Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Collect all paths + labels, then stratified split ─────────────────────
all_paths, all_labels = [], []
for cls in TARGET_CLASSES:
    imgs = get_images(os.path.join(UNIFIED_DIR, cls))
    all_paths.extend(imgs)
    all_labels.extend([CLASS_TO_IDX[cls]] * len(imgs))

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels, dtype=np.int32)

# 70% train / 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    all_paths, all_labels,
    test_size=(VAL_SPLIT + TEST_SPLIT),
    random_state=SEED, stratify=all_labels
)

# 15% val / 15% test (equal split of temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=SEED, stratify=y_temp
)

total_n = len(all_paths)
print('Train / Val / Test split (stratified by class):')
print(f'  Train : {len(X_train):>6}  ({len(X_train)/total_n*100:.1f}%)')
print(f'  Val   : {len(X_val):>6}  ({len(X_val)/total_n*100:.1f}%)')
print(f'  Test  : {len(X_test):>6}  ({len(X_test)/total_n*100:.1f}%)')
print(f'  Total : {total_n:>6}')


In [ ]:
# ── Preprocessing & Augmentation Pipeline ─────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(path, label):
    raw   = tf.io.read_file(path)
    img   = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img   = tf.image.resize(img, IMG_SIZE)
    img   = tf.cast(img, tf.float32) / 255.0
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)
    img = tf.image.random_hue(img, 0.05)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=min(len(paths), 8000), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(X_train, y_train, training=True)
val_ds   = make_dataset(X_val,   y_val,   training=False)
test_ds  = make_dataset(X_test,  y_test,  training=False)

steps_per_epoch = math.ceil(len(X_train) / BATCH_SIZE)
val_steps       = math.ceil(len(X_val)   / BATCH_SIZE)
test_steps      = math.ceil(len(X_test)  / BATCH_SIZE)

# Verify one batch
for imgs_batch, lbl_batch in train_ds.take(1):
    print(f'Batch shape  : {imgs_batch.shape}')
    print(f'Labels shape : {lbl_batch.shape}')
    print(f'Pixel range  : [{imgs_batch.numpy().min():.3f}, {imgs_batch.numpy().max():.3f}]')
print(f'Steps/epoch  : {steps_per_epoch}')
print(f'Val steps    : {val_steps}')


## Step 7 — Custom Architecture

```
Input (None, 256, 256, 3)
│
├─ Inception_cell(16)   → [3×3 Conv ⊕ 5×5 Conv ⊕ 7×7 Conv] → Add  → (256,256,16)
│   BatchNorm → MaxPool(2×2)                                          → (128,128,16)
│
├─ Inception_cell(32)   → same pattern                                → (128,128,32)
│   BatchNorm → MaxPool(2×2)                                          → ( 64, 64,32)
│
├─ SkipConnection(64)   → Conv7×7 → BN → Concat(input)               → ( 64, 64,96)
│   MaxPool(2×2)                                                      → ( 32, 32,96)
│
├─ SkipConnection(128)  → Conv7×7 → BN → Concat(input)               → ( 32, 32,224)
│   MaxPool(2×2)                                                      → ( 16, 16,224)
│
└─ Flatten(57344) → Dense(1024) → Dense(256) → Dropout(0.5)
                 → Dense(128)  → Dense(64)  → Dense(24, softmax)
```


In [ ]:
class CosineAnnealingScheduler(LearningRateScheduler):
    """
    LR decays from initial_learning_rate to ~0 following a cosine curve.
    lr(epoch) = 0.5 * lr0 * (1 + cos(pi * epoch / T))
    """
    def __init__(self, initial_learning_rate, total_epochs):
        self.total_epochs = total_epochs
        self.initial_lr   = initial_learning_rate
        super().__init__(self._schedule, verbose=0)

    def _schedule(self, epoch, lr):
        return 0.5 * self.initial_lr * (
            1 + math.cos(math.pi * epoch / self.total_epochs)
        )

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'initial_lr': self.initial_lr,
                    'total_epochs': self.total_epochs})
        return cfg


# Preview the schedule
ep_range = range(EPOCHS)
lrs      = [0.5*INITIAL_LR*(1+math.cos(math.pi*e/EPOCHS)) for e in ep_range]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(ep_range, lrs, color='#5B7FDE', lw=2)
ax.fill_between(ep_range, lrs, alpha=0.1, color='#5B7FDE')
ax.set_xlabel('Epoch'); ax.set_ylabel('Learning Rate')
ax.set_title(f'Cosine Annealing LR  (init={INITIAL_LR}, T={EPOCHS})',
             fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'lr_schedule.png'), dpi=150, bbox_inches='tight')
plt.show()
print('CosineAnnealingScheduler defined.')


In [ ]:
class Inception_cell(Layer):
    """
    Three parallel convolutions (3×3, 5×5, 7×7), all with `num_filters` output
    channels, combined via element-wise Add (not Concat).

    Parameters:
    -----------
    num_filters : int
        Number of filters for each branch (and the output depth).
    activation  : str
        Activation function applied inside each Conv2D.
    padding     : str
        'same' keeps spatial dimensions unchanged.

    Parameter count (input_channels = C):
        3×3 branch : (3×3×C×F) + F
        5×5 branch : (5×5×C×F) + F
        7×7 branch : (7×7×C×F) + F
    """
    def __init__(self, num_filters, activation='relu', padding='same', **kwargs):
        super().__init__(**kwargs)
        self.num_filters = num_filters
        self.activation  = activation
        self.padding     = padding
        self.conv3 = Conv2D(num_filters, (3,3), activation=activation, padding=padding)
        self.conv5 = Conv2D(num_filters, (5,5), activation=activation, padding=padding)
        self.conv7 = Conv2D(num_filters, (7,7), activation=activation, padding=padding)

    def call(self, x, training=False):
        x1 = self.conv3(x)
        x2 = self.conv5(x)
        x3 = self.conv7(x)
        return Add()([x1, x2, x3])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_filters': self.num_filters,
                    'activation' : self.activation,
                    'padding'    : self.padding})
        return cfg

print('Inception_cell defined.')


In [ ]:
class SkipConnection(Layer):
    """
    DenseNet-style skip connection:
        Conv2D(7×7) → BatchNormalization → Concatenate(input)

    Output depth = num_filters + input_channels
    (The original signal is fully preserved, unlike ResNet's Add.)

    Parameter count (input_channels = C, num_filters = F):
        Conv 7×7 : (7×7×C×F) + F
        BN       : 4×F  (gamma, beta, moving_mean, moving_var)
    """
    def __init__(self, num_filters, kernel_size=7, activation='relu',
                 padding='same', **kwargs):
        super().__init__(**kwargs)
        self.num_filters = num_filters
        self.kernel_size = kernel_size
        self.activation  = activation
        self.padding     = padding
        # kernel_size stored but always 7×7 (matches paper implementation)
        self.conv = Conv2D(num_filters, (7,7), activation=activation, padding=padding)
        self.bn   = BatchNormalization()

    def call(self, x, training=False):
        out = self.conv(x)
        out = self.bn(out, training=training)
        return Concatenate(axis=-1)([out, x])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_filters': self.num_filters,
                    'kernel_size': self.kernel_size,
                    'activation' : self.activation,
                    'padding'    : self.padding})
        return cfg

print('SkipConnection defined.')


In [ ]:
def build_model(input_shape=IMG_SHAPE, num_classes=NUM_CLASSES,
                dropout_rate=DROPOUT_RATE):
    inputs = Input(shape=input_shape, name='input')

    # ── Inception Block 1 ─────────────────────────────────────────────────
    x = Inception_cell(num_filters=16, name='inception_cell')(inputs)   # 256×256×16
    x = BatchNormalization(name='bn_inc_1')(x)
    x = MaxPooling2D((2,2), name='pool_1')(x)                           # 128×128×16

    # ── Inception Block 2 ─────────────────────────────────────────────────
    x = Inception_cell(num_filters=32, name='inception_cell_1')(x)      # 128×128×32
    x = BatchNormalization(name='bn_inc_2')(x)
    x = MaxPooling2D((2,2), name='pool_2')(x)                           #  64× 64×32

    # ── Skip Connection 1 ─────────────────────────────────────────────────
    x = SkipConnection(num_filters=64,  name='skip_connection')(x)      #  64× 64×96
    x = MaxPooling2D((2,2), name='pool_3')(x)                           #  32× 32×96

    # ── Skip Connection 2 ─────────────────────────────────────────────────
    x = SkipConnection(num_filters=128, name='skip_connection_1')(x)    #  32× 32×224
    x = MaxPooling2D((2,2), name='pool_4')(x)                           #  16× 16×224

    # ── Classifier Head ───────────────────────────────────────────────────
    x = Flatten(name='flatten')(x)                                       # 57,344
    x = Dense(1024, activation='relu', name='fc_1024')(x)
    x = Dense(256,  activation='relu', name='fc_256')(x)
    x = Dropout(dropout_rate, name='dropout')(x)
    x = Dense(128,  activation='relu', name='fc_128')(x)
    x = Dense(64,   activation='relu', name='fc_64')(x)
    outputs = Dense(num_classes, activation='softmax', name='output')(x)

    return Model(inputs, outputs, name='PlantDiseaseNet')


model = build_model()
model.summary(line_length=90)


In [ ]:
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=INITIAL_LR),
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy']
)
print(f'Model compiled.')
print(f'Total parameters : {model.count_params():,}')


In [ ]:
ckpt_path = os.path.join(CKPT_DIR, 'best_model.weights.h5')

callbacks = [
    CosineAnnealingScheduler(
        initial_learning_rate = INITIAL_LR,
        total_epochs          = EPOCHS
    ),
    ModelCheckpoint(
        filepath          = ckpt_path,
        monitor           = 'val_accuracy',
        save_best_only    = True,
        save_weights_only = True,
        verbose           = 1
    ),
    EarlyStopping(
        monitor              = 'val_loss',
        patience             = 10,
        restore_best_weights = True,
        verbose              = 1
    ),
]

print('Callbacks:')
for cb in callbacks:
    print(f'  {type(cb).__name__}')


In [ ]:
print(f'Starting training for up to {EPOCHS} epochs ...')
print(f'Best weights will be saved to: {ckpt_path}')
print('-' * 70)

history = model.fit(
    train_ds,
    epochs          = EPOCHS,
    validation_data = val_ds,
    callbacks       = callbacks,
    verbose         = 1
)

# Save full model (H5 for compatibility with custom layers)
h5_path = os.path.join(BASE_DIR, 'plant_disease_model.h5')
model.save(h5_path)
print(f'\nModel saved: {h5_path}')


In [ ]:
hist   = history.history
n_ep   = range(1, len(hist['loss']) + 1)
lrs_ep = [0.5*INITIAL_LR*(1+math.cos(math.pi*e/EPOCHS))
          for e in range(len(hist['loss']))]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, train_key, val_key, title in [
    (axes[0], 'accuracy', 'val_accuracy', 'Accuracy'),
    (axes[1], 'loss',     'val_loss',     'Loss (Categorical Cross-Entropy)'),
]:
    ax.plot(n_ep, hist[train_key], label='Train', color='#5B7FDE', lw=2)
    ax.plot(n_ep, hist[val_key],   label='Val',   color='#E86B6B', lw=2)
    ax.set_xlabel('Epoch'); ax.set_title(title, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

axes[2].plot(n_ep, lrs_ep, color='#52B788', lw=2)
axes[2].fill_between(n_ep, lrs_ep, alpha=0.1, color='#52B788')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].set_title('Learning Rate (Cosine Annealing)', fontweight='bold')
axes[2].grid(True, alpha=0.3)

fig.suptitle('Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Best val accuracy : {max(hist["val_accuracy"]):.4f}')
print(f'Best val loss     : {min(hist["val_loss"]):.4f}')
print(f'Epochs run        : {len(hist["loss"])}')


## Step 9 — Evaluation on Test Set

Metrics: **Accuracy · Weighted Precision · Weighted Recall · Weighted F1-Score · Confusion Matrix · Per-class Report**


In [ ]:
# Load best saved weights before evaluating
if os.path.exists(ckpt_path):
    model.load_weights(ckpt_path)
    print(f'Loaded best weights from: {ckpt_path}')

print('Predicting on test set ...')
y_pred_probs = model.predict(test_ds, steps=test_steps, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)

# Extract true labels in same order as test_ds
y_true = np.concatenate([y for _, y in test_ds], axis=0)
y_true = y_true[:len(y_pred)]   # align if last batch was partial

print(f'Test samples    : {len(y_true)}')
print(f'Prediction shape: {y_pred.shape}')


In [ ]:
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_true,   y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_true,       y_pred, average='weighted', zero_division=0)

sep = '=' * 52
print(sep)
print('          TEST SET EVALUATION RESULTS')
print(sep)
print(f'  Accuracy           : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision (wt avg) : {prec:.4f}')
print(f'  Recall    (wt avg) : {rec:.4f}')
print(f'  F1-Score  (wt avg) : {f1:.4f}')
print(sep)

metrics_dict = {'Accuracy':acc,'Precision':prec,'Recall':rec,'F1-Score':f1}
fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar(metrics_dict.keys(), metrics_dict.values(),
              color=['#5B7FDE','#52B788','#E8845A','#E86B6B'],
              edgecolor='white', width=0.5)
for bar, val in zip(bars, metrics_dict.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, 1.08)
ax.set_ylabel('Score'); ax.grid(True, axis='y', alpha=0.3)
ax.set_title('Test Set Performance', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR,'test_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

short = [
    c.replace('Cotton_','C.').replace('Wheat_','W.')
     .replace('Potato_','P.').replace('Bell_Pepper_','BP.')
     .replace('Tomato_','T.').replace('_',' ')
    for c in TARGET_CLASSES
]

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
for ax, data, fmt, cmap, title in [
    (axes[0], cm,      'd',    'Blues',  'Confusion Matrix (counts)'),
    (axes[1], cm_norm, '.2f',  'RdYlGn', 'Confusion Matrix (normalised)'),
]:
    sns.heatmap(data, ax=ax, cmap=cmap, fmt=fmt, annot=True,
                xticklabels=short, yticklabels=short,
                linewidths=0.3, linecolor='white',
                **({'vmin':0,'vmax':1} if fmt=='.2f' else {}))
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('True',      fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR,'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
report_str  = classification_report(
    y_true, y_pred, target_names=TARGET_CLASSES, digits=4, zero_division=0
)
report_dict = classification_report(
    y_true, y_pred, target_names=TARGET_CLASSES,
    output_dict=True, zero_division=0
)
print(report_str)

# Save text report
with open(os.path.join(LOG_DIR,'classification_report.txt'), 'w') as f:
    f.write(report_str)

# Per-class F1 bar chart
class_f1s = [report_dict[c]['f1-score'] for c in TARGET_CLASSES]
bar_cols   = ['#52B788' if v>=0.9 else '#F5C842' if v>=0.7 else '#E86B6B'
              for v in class_f1s]

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(NUM_CLASSES), class_f1s, color=bar_cols, edgecolor='white')
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(short, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('F1-Score'); ax.set_ylim(0, 1.05)
ax.set_title('Per-Class F1-Score', fontweight='bold', fontsize=12)
ax.axhline(np.mean(class_f1s), color='black', ls='--', lw=1, alpha=0.6,
           label=f'Mean F1 = {np.mean(class_f1s):.3f}')
legend_elems = [
    Patch(facecolor='#52B788', label='F1 >= 0.90'),
    Patch(facecolor='#F5C842', label='0.70 <= F1 < 0.90'),
    Patch(facecolor='#E86B6B', label='F1 < 0.70'),
]
ax.legend(handles=legend_elems, loc='lower right', fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR,'per_class_f1.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
random.seed(SEED)
n_show    = 16
sample_i  = random.sample(range(len(X_test)), n_show)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
axes_flat = axes.flatten()

for plot_i, test_i in enumerate(sample_i):
    img_path  = X_test[test_i]
    true_lbl  = int(y_test[test_i])

    raw       = tf.io.read_file(img_path)
    img_t     = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img_t     = tf.image.resize(img_t, IMG_SIZE)
    img_disp  = img_t.numpy().astype('uint8')
    img_norm  = tf.cast(img_t, tf.float32)[tf.newaxis] / 255.0

    probs     = model(img_norm, training=False).numpy()[0]
    pred_cls  = int(np.argmax(probs))
    conf      = probs[pred_cls] * 100
    correct   = (true_lbl == pred_cls)

    true_name = IDX_TO_CLASS[true_lbl].replace('_',' ')
    pred_name = IDX_TO_CLASS[pred_cls].replace('_',' ')
    color     = '#2d6a4f' if correct else '#c1121f'

    ax = axes_flat[plot_i]
    ax.imshow(img_disp)
    ax.set_title(f'True: {true_name}\nPred: {pred_name} ({conf:.1f}%)',
                 fontsize=7.5, color=color, pad=3)
    for sp in ax.spines.values():
        sp.set_edgecolor(color); sp.set_linewidth(2.5)
    ax.tick_params(left=False, bottom=False,
                   labelleft=False, labelbottom=False)

fig.suptitle('Sample Predictions   green = correct  |  red = incorrect',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR,'sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Save class mapping ────────────────────────────────────────────────────
mapping_path = os.path.join(BASE_DIR, 'class_mapping.json')
with open(mapping_path, 'w') as f:
    json.dump({'idx_to_class': IDX_TO_CLASS,
               'class_to_idx': CLASS_TO_IDX}, f, indent=2)
print(f'Class mapping saved : {mapping_path}')

# ── Final summary ────────────────────────────────────────────────────────
print()
print('=' * 52)
print('              FINAL RESULTS')
print('=' * 52)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print('=' * 52)
print(f'  Model     : {h5_path}')
print(f'  Logs      : {LOG_DIR}')
print()

# ── Reload snippet ────────────────────────────────────────────────────────
print('=== HOW TO RELOAD THE MODEL ===')
print('''
from tensorflow.keras.models import load_model
from tensorflow.keras.saving import custom_object_scope

with custom_object_scope({
    'Inception_cell'          : Inception_cell,
    'SkipConnection'          : SkipConnection,
    'CosineAnnealingScheduler': CosineAnnealingScheduler,
}):
    model = load_model('plant_disease_model.h5')
''')
